#### Imports

In [1]:
import pandas as pd
import numpy as np
import time
import pyodbc
import pickle
import warnings
import os
from tensorflow.keras.models import load_model
from collections import deque
import winsound

# Suppress Deprecation and FutureWarnings for a cleaner live console output
warnings.filterwarnings('ignore')   

print("--- INITIALIZING FACTORY INFERENCE ENGINE ---")

--- INITIALIZING FACTORY INFERENCE ENGINE ---


#### Azure SQL & Path Configuration

In [2]:
# AZURE SQL CONFIGURATION
AZURE_SERVER = 'kreseakreiotprdsrv.database.windows.net'
AZURE_DATABASE = 'KRESEAKREIOTPRD'
AZURE_USERNAME = 'IOTAdmin'
AZURE_PASSWORD = 'oKuvodump5JNG7dM' 
AZURE_TABLE_NAME = 'MBP_ControllerData' 

# PATHS TO TRAINED ARTIFACTS
MODEL_PATH = 'best_overlock_lstm.keras'
SCALER_PATH = 'scaler.pkl'
ENCODER_PATH = 'encoder.pkl'

# Select Target Machine
SELECTED_MACHINE = input("Enter the Machine Serial Number to analyze (e.g., 0521760): ").strip()
print(f"Target Machine Set To: {SELECTED_MACHINE}")

# Connection String
driver = '{ODBC Driver 17 for SQL Server}'
conn_str = f'DRIVER={driver};SERVER={AZURE_SERVER};PORT=1433;DATABASE={AZURE_DATABASE};UID={AZURE_USERNAME};PWD={AZURE_PASSWORD}'

Target Machine Set To: YN68884


#### Load Model & Maintenance Solutions

In [3]:
# MAINTENANCE KNOWLEDGE BASE (Industrial Best Practices)
maintenance_solutions = {
    "Needle Breakages": "SOLUTION: Immediately replace the needle. Check needle guard alignment and needle-to-looper timing.",
    "High Foot Pressure": "SOLUTION: Reduce presser foot pressure dial. Check for fabric marking or feeding resistance.",
    "Cut/Needle Hole": "SOLUTION: Check needle point for damage/burrs. Ensure needle size is appropriate for the fabric GSM.",
    "Thread Breakages": "SOLUTION: Re-thread the machine. Check thread quality, tension discs, and look for sharp edges on the looper.",
    "Pneumatic Issues": "SOLUTION: Check air compressor pressure. Inspect pneumatic hoses for leaks or kinks in the auto-cutter unit.",
    "Thread Jamming": "SOLUTION: Clear the bobbin/looper area. Remove lint from feed dogs and check the threading path.",
    "Code Uneven": "SOLUTION: Synchronize feed dog movement. Adjust differential feed ratio for the specific fabric type.",
    "Roping": "SOLUTION: Adjust the differential feed. Check presser foot alignment to ensure even pressure across the seam.",
    "Oil Mark": "SOLUTION: Check for oil leaks in the needle bar area. Clean the machine and check oil pump pressure.",
    "Skip Stitches/Slip": "SOLUTION: Adjust needle-to-looper clearance. Check needle depth and ensure the needle is not bent.",
    "Gathering/Puckering": "SOLUTION: Decrease thread tension. Increase the differential feed ratio and check for blunt needles.",
    "Waveness": "SOLUTION: Adjust the differential feed. Ensure the operator is not pulling the fabric too tightly.",
    "Binding/Seam Open": "SOLUTION: Increase stitch density. Check folder/binder alignment and ensure consistent seam allowance.",
    "Blade Broken": "SOLUTION: STOP MACHINE. Replace upper and lower trimming blades immediately. Check blade timing.",
    "Healthy": "STATUS: Machine operating within normal parameters. No maintenance required."
}

try:
    # Load the specific artifacts used in LSTM.ipynb
    model = load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    with open(ENCODER_PATH, 'rb') as f:
        encoder = pickle.load(f)
    print("✅ AI Model, Translators, and Maintenance Solutions Loaded.")
except Exception as e:
    print(f"❌ ERROR LOADING FILES: {e}")

✅ AI Model, Translators, and Maintenance Solutions Loaded.


#### Logic Engine: Live Feature Extraction(10Hz Banding)

In [4]:
def extract_live_features(df):
    df = df[df['machineVibration'].astype(str).str.startswith('10')].copy()
    if df.empty: return None

    vib_records = []
    for val in df['machineVibration']:
        vib_dict = {}
        parts = str(val).split(',')
        try:
            for i in range(0, len(parts)-1, 2):
                f_start = int(parts[i])
                f_end = f_start + 10
                vib_dict[f"{f_start}-{f_end}Hz"] = int(parts[i+1])
        except: pass
        vib_records.append(vib_dict)
    
    vib_df = pd.DataFrame(vib_records)
    expected_cols = [f"{i}-{i+10}Hz" for i in range(10, 610, 10)]
    vib_df = vib_df.reindex(columns=expected_cols, fill_value=0)
    
    elec_cols = ['machineVoltageMean', 'machineVoltageMin', 'machineVoltageMax',
                 'machineCurrentMean', 'machineCurrentMin', 'machineCurrentMax']
    elec_df = df[elec_cols].ffill().fillna(0)
    
    return pd.concat([vib_df, elec_df.reset_index(drop=True)], axis=1)

#### The Live Monitoring Loop

In [6]:
last_processed_time = None
TIME_STEPS = 5

try:
    print(f"🔄 Connecting to Azure SQL Database...")
    with pyodbc.connect(conn_str) as conn:
        print(f"✅ Monitoring Machine: {SELECTED_MACHINE}")
        
        # Pulls latest records for temporal analysis
        query = f"SELECT TOP {TIME_STEPS} * FROM {AZURE_TABLE_NAME} WHERE machineSerialNumber = '{SELECTED_MACHINE}' ORDER BY dateTime DESC"
        
        while True:
            df_live = pd.read_sql(query, conn)
            
            if not df_live.empty and len(df_live) >= TIME_STEPS:
                db_raw_ts = df_live['dateTime'].iloc[0]
                
                if db_raw_ts != last_processed_time:
                    last_processed_time = db_raw_ts
                    
                    # Time synchronization for SL (UTC + 5:30)
                    sl_record_time = pd.to_datetime(db_raw_ts) + pd.Timedelta(hours=5, minutes=30)
                    current_local_time = pd.Timestamp.now()
                    
                    # Prepare 5-step window for LSTM inference
                    df_window_raw = df_live.iloc[::-1].reset_index(drop=True)
                    df_features = extract_live_features(df_window_raw)
                    
                    if df_features is not None and len(df_features) == TIME_STEPS:
                        # Scale and reshape using scaler.pkl (1, 5, 66)
                        X_scaled = scaler.transform(df_features.values).reshape(1, TIME_STEPS, -1)
                        probs = model.predict(X_scaled, verbose=0)[0]
                        max_idx = np.argmax(probs)
                        confidence = probs[max_idx]
                        predicted_state = encoder.inverse_transform([max_idx])[0]
                        
                        # Enhanced Terminal Output for Viva Presentation
                        print("\n" + "="*55)
                        print(f"AZURE RECORD (UTC) : {str(db_raw_ts).split('.')[0]}") 
                        print(f"RECORD TIME (SL)   : {sl_record_time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print(f"CURRENT LOCAL TIME : {current_local_time.strftime('%Y-%m-%d %H:%M:%S')}")
                        print("-" * 55)
                        print(f"PREDICTED STATE    : {predicted_state} ({confidence*100:.2f}%)")
                        
                        # Apply 75% Confidence Threshold Logic for Anomaly Detection
                        if predicted_state != "Healthy" and confidence >= 0.75:
                            print(f"🚨 ALERT: {predicted_state} DETECTED")
                            for _ in range(3): winsound.Beep(1000, 500) # Triple beep for confirmed alert
                            
                            # Display mapped maintenance solution
                            if predicted_state in maintenance_solutions:
                                print(f"🛠️ {maintenance_solutions[predicted_state]}")
                                
                        elif confidence < 0.75:
                            # Anomaly suspected but below confidence threshold
                            print(f"⚠️UNKNOWN STATE: Anomaly suspected. Machine behaving erratically.")
                            print(f"Predicted Anomaly: {predicted_state}")
                            for _ in range(2): winsound.Beep(1000, 500) # Double beep for suspicion
                            
                            # Provide solution for suspected state as a precaution
                            if predicted_state in maintenance_solutions:
                                print(f"📝{maintenance_solutions[predicted_state]}")
                        else:
                            print("✅ STATUS: Machine operating normally.")
                        print("="*55)
                    else:
                        print(f"Skipping: Invalid vibration data pattern at {sl_record_time.strftime('%H:%M:%S')}", end="\r")
            
            time.sleep(1) # Frequency of polling Azure database

except KeyboardInterrupt:
    print("\n🛑 Monitoring stopped by user.")
except Exception as e:
    print(f"\n❌ SCRIPT ERROR: {e}")


AZURE RECORD (UTC) : 2026-03-11 06:21:36
RECORD TIME (SL)   : 2026-03-11 11:51:36
CURRENT LOCAL TIME : 2026-03-11 11:51:36
-------------------------------------------------------
PREDICTED STATE    : High Foot Pressure (50.09%)
⚠️UNKNOWN STATE: Anomaly suspected. Machine behaving erratically.
Predicted Anomaly: High Foot Pressure
📝SOLUTION: Reduce presser foot pressure dial. Check for fabric marking or feeding resistance.

AZURE RECORD (UTC) : 2026-03-11 06:36:53
RECORD TIME (SL)   : 2026-03-11 12:06:53
CURRENT LOCAL TIME : 2026-03-11 12:06:54
-------------------------------------------------------
PREDICTED STATE    : Healthy (51.79%)
⚠️UNKNOWN STATE: Anomaly suspected. Machine behaving erratically.
Predicted Anomaly: Healthy
📝STATUS: Machine operating within normal parameters. No maintenance required.

AZURE RECORD (UTC) : 2026-03-11 06:39:00
RECORD TIME (SL)   : 2026-03-11 12:09:00
CURRENT LOCAL TIME : 2026-03-11 12:09:01
-------------------------------------------------------
PRED